# Python typing 与 Pydantic 实战教程

本教程面向具备 Python 基础的开发者，系统学习：

- Python 类型提示与静态类型检查
- `typing` 中的常用类型工具
- Pydantic v2 的数据解析、校验与序列化
- `typing` 与 Pydantic 的组合设计
- 配置、验证器、泛型、递归模型、`TypeAdapter`
- API 请求模型与业务配置模型实战
- 常见错误、工程规范与练习

> 类型提示主要服务于 IDE、静态类型检查器和开发者阅读，Python 运行时通常不会自动强制执行类型约束。Pydantic 则会在运行时读取类型标注并完成数据验证与转换。

建议环境：Python 3.10+，Pydantic 2.x。

## 环境准备


In [ ]:
import sys
import pydantic

print("Python:", sys.version)
print("Pydantic:", pydantic.__version__)

## typing 的作用

类型标注不会改变函数的基本运行方式，但可以：

- 提升代码可读性
- 为 IDE 提供自动补全
- 提前发现类型错误
- 作为 Pydantic、FastAPI 等框架的结构描述
- 支持大型项目中的接口约束

下面的函数声明了参数和返回值类型。

In [ ]:
def add(x: int, y: int) -> int:
    return x + y

result = add(10, 20)
print(result)

# Python 默认不会阻止这种调用，是否报错取决于函数内部运算。
print(add("hello ", "typing"))

## 基础类型标注

Python 3.9+ 推荐直接使用内置容器类型：

- `list[str]`
- `dict[str, int]`
- `tuple[int, str]`
- `set[int]`

旧式的 `typing.List`、`typing.Dict` 仍可能出现在历史项目中，但新代码通常优先使用内置泛型语法。

In [ ]:
name: str = "Alice"
age: int = 25
score: float = 95.5
is_active: bool = True
tags: list[str] = ["python", "pydantic"]
profile: dict[str, str] = {"city": "Tokyo", "role": "developer"}
point: tuple[int, int] = (10, 20)
unique_ids: set[int] = {1, 2, 3}

print(name, age, tags, profile)

## Union、Optional 与 None

`A | B` 表示值可以是 A 或 B。

`str | None` 表示字符串或空值，等价于 `Optional[str]`。需要注意：

- “允许为 None”不等于“可以省略参数”
- 是否可以省略，取决于是否提供默认值

In [ ]:
from typing import Optional, Union

def normalize_id(value: int | str) -> str:
    return str(value).strip()

def greet(name: str | None = None) -> str:
    return f"你好，{name or '访客'}"

legacy_value: Union[int, str] = 100
legacy_optional: Optional[str] = None

print(normalize_id(1001))
print(normalize_id("  A-100  "))
print(greet())

## Literal、Final 与 ClassVar

- `Literal`：限制为几个固定字面量
- `Final`：表达变量不应被重新赋值，主要由静态检查器检查
- `ClassVar`：声明类变量，而不是实例字段

In [ ]:
from typing import ClassVar, Final, Literal

Environment = Literal["dev", "test", "prod"]

MAX_RETRY: Final[int] = 3

class Service:
    service_name: ClassVar[str] = "user-service"

    def __init__(self, environment: Environment) -> None:
        self.environment = environment

service = Service("dev")
print(service.service_name, service.environment, MAX_RETRY)

## TypeAlias 与可读性

复杂类型可以抽取为类型别名。Python 3.10/3.11 项目可使用赋值语法；Python 3.12+ 还支持 `type` 语句。

In [ ]:
from typing import TypeAlias

UserId: TypeAlias = int
JSONPrimitive: TypeAlias = str | int | float | bool | None
JSONValue: TypeAlias = JSONPrimitive | list["JSONValue"] | dict[str, "JSONValue"]

def load_user(user_id: UserId) -> dict[str, JSONValue]:
    return {
        "id": user_id,
        "name": "Alice",
        "roles": ["admin", "editor"],
        "active": True,
    }

print(load_user(1))

## Callable 与函数类型

`Callable[[参数类型...], 返回值类型]` 用于描述函数、回调和策略对象。

In [ ]:
from collections.abc import Callable

Formatter = Callable[[str], str]

def upper_formatter(text: str) -> str:
    return text.upper()

def render(value: str, formatter: Formatter) -> str:
    return formatter(value)

print(render("hello typing", upper_formatter))

## TypedDict：描述字典结构

`TypedDict` 适合：

- 已有代码必须继续使用字典
- 描述 JSON 风格结构
- 只需要静态类型检查，不需要运行时模型实例

它本身通常不会在运行时验证字典内容。

In [ ]:
from typing import NotRequired, Required, TypedDict

class UserPayload(TypedDict):
    id: Required[int]
    name: Required[str]
    email: NotRequired[str]
    age: NotRequired[int]

payload: UserPayload = {
    "id": 1,
    "name": "Alice",
    "email": "alice@example.com",
}

print(payload)

## Protocol：面向行为编程

`Protocol` 描述对象需要具备哪些方法，而不要求继承同一个父类。适合插件、存储层、模型客户端和依赖注入。

In [ ]:
from typing import Protocol

class Repository(Protocol):
    def get(self, item_id: int) -> dict[str, object] | None:
        ...

class MemoryRepository:
    def __init__(self) -> None:
        self.data = {1: {"id": 1, "name": "Alice"}}

    def get(self, item_id: int) -> dict[str, object] | None:
        return self.data.get(item_id)

def find_name(repo: Repository, item_id: int) -> str | None:
    item = repo.get(item_id)
    return str(item["name"]) if item else None

print(find_name(MemoryRepository(), 1))

## TypeVar 与泛型

泛型允许一个类或函数在保持类型关联的同时支持多种数据类型。

In [ ]:
from typing import Generic, TypeVar

T = TypeVar("T")

class Page(Generic[T]):
    def __init__(self, items: list[T], total: int) -> None:
        self.items = items
        self.total = total

def first_or_none(items: list[T]) -> T | None:
    return items[0] if items else None

user_page = Page[dict[str, object]](
    items=[{"id": 1, "name": "Alice"}],
    total=1,
)

print(user_page.items)
print(first_or_none([1, 2, 3]))

## Annotated：给类型附加元数据

`Annotated[T, metadata...]` 保留基础类型 T，并为框架提供额外元数据。Pydantic 常使用它声明长度、范围、描述和验证规则。

In [ ]:
from typing import Annotated, get_args, get_origin

UserName = Annotated[str, "用户名，长度应受限制"]

print(get_origin(UserName))
print(get_args(UserName))

# Pydantic 基础

Pydantic 的核心类是 `BaseModel`。它会根据字段的类型标注：

- 接收字典或关键字参数
- 验证数据类型和约束
- 在允许的情况下转换输入
- 生成结构化错误
- 序列化为字典或 JSON
- 生成 JSON Schema

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    age: int | None = None
    tags: list[str] = []

user = User(id="1001", name="Alice", age="25", tags=["python", "api"])

print(user)
print(type(user.id), type(user.age))

## 不要使用可变默认值吗？

普通 Python 类和函数中应避免直接使用可变默认值。Pydantic 会对常见的非哈希默认值进行复制，但工程中仍推荐使用 `Field(default_factory=...)`，意图更加清晰。

In [ ]:
from pydantic import Field

class SafeUser(BaseModel):
    id: int
    tags: list[str] = Field(default_factory=list)
    metadata: dict[str, str] = Field(default_factory=dict)

u1 = SafeUser(id=1)
u2 = SafeUser(id=2)
u1.tags.append("admin")

print(u1.tags)
print(u2.tags)

## ValidationError 与结构化错误

验证失败时，Pydantic 抛出 `ValidationError`。常用查看方式：

- `str(exc)`：适合日志
- `exc.errors()`：结构化错误列表
- `exc.json()`：JSON 字符串

In [ ]:
from pydantic import ValidationError

class Product(BaseModel):
    id: int
    name: str
    price: float

try:
    Product(id="not-an-int", name="Keyboard", price="unknown")
except ValidationError as exc:
    print(exc)
    print(exc.errors())

## Field 字段约束

`Field` 可以声明：

- 数值范围：`gt`、`ge`、`lt`、`le`
- 字符串长度：`min_length`、`max_length`
- 正则模式：`pattern`
- 别名：`alias`
- 描述、示例、标题等 Schema 信息

In [ ]:
from typing import Annotated
from pydantic import Field

PositivePrice = Annotated[float, Field(gt=0)]
Sku = Annotated[str, Field(min_length=3, max_length=20, pattern=r"^[A-Z0-9-]+$")]

class ConstrainedProduct(BaseModel):
    sku: Sku
    name: Annotated[str, Field(min_length=1, max_length=100)]
    price: PositivePrice
    stock: Annotated[int, Field(ge=0)] = 0

product = ConstrainedProduct(
    sku="KB-001",
    name="Mechanical Keyboard",
    price=499.0,
    stock=20,
)

print(product)

## 嵌套模型

复杂业务结构应该拆分为多个小模型。嵌套字段可以接收字典，Pydantic 会递归构造子模型。

In [ ]:
class Address(BaseModel):
    province: str
    city: str
    detail: str

class Customer(BaseModel):
    id: int
    name: str
    address: Address

customer = Customer(
    id=1,
    name="Alice",
    address={
        "province": "Tokyo",
        "city": "Shinjuku",
        "detail": "1-2-3",
    },
)

print(customer)
print(type(customer.address))

## model_dump 与 model_dump_json

Pydantic v2 常用序列化方法：

- `model_dump()`：返回 Python 字典
- `model_dump_json()`：返回 JSON 字符串
- `exclude_none=True`：忽略空值
- `include` / `exclude`：控制字段集合
- `by_alias=True`：使用字段别名

In [ ]:
class Profile(BaseModel):
    user_id: int
    nickname: str
    bio: str | None = None

profile = Profile(user_id=1, nickname="alice")

print(profile.model_dump())
print(profile.model_dump(exclude_none=True))
print(profile.model_dump(include={"user_id", "nickname"}))
print(profile.model_dump_json(indent=2))

## model_validate 与 model_validate_json

Pydantic v2 使用：

- `Model.model_validate(data)`：验证 Python 对象
- `Model.model_validate_json(json_text)`：验证 JSON 字符串

In [ ]:
raw_dict = {"user_id": "100", "nickname": "alice"}
raw_json = '{"user_id": "101", "nickname": "bob"}'

p1 = Profile.model_validate(raw_dict)
p2 = Profile.model_validate_json(raw_json)

print(p1)
print(p2)

## ConfigDict 模型配置

常见配置项：

- `extra="forbid"`：拒绝未声明字段
- `strict=True`：尽量禁止隐式类型转换
- `frozen=True`：模型不可重新赋值
- `validate_assignment=True`：赋值时重新验证
- `populate_by_name=True`：允许使用字段名填充有别名的字段
- `str_strip_whitespace=True`：自动去除字符串两端空白

In [ ]:
from pydantic import ConfigDict

class StrictAccount(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        validate_assignment=True,
        str_strip_whitespace=True,
    )

    username: str
    age: int

account = StrictAccount(username="  alice  ", age=20)
print(account)

try:
    account.age = -1
    print(account)
except ValidationError as exc:
    print(exc)

上面的 `validate_assignment=True` 只会重新执行已有类型与字段约束。为了限制年龄范围，需要显式添加 `Field` 约束。

In [ ]:
class ValidatedAccount(BaseModel):
    model_config = ConfigDict(validate_assignment=True)

    username: Annotated[str, Field(min_length=3)]
    age: Annotated[int, Field(ge=0, le=150)]

account = ValidatedAccount(username="alice", age=20)

try:
    account.age = -1
except ValidationError as exc:
    print(exc)

## 严格模式

Pydantic 默认可能进行合理的数据转换，例如将 `"42"` 转为整数 `42`。严格模式适用于不希望发生隐式转换的接口边界。

In [ ]:
class LooseModel(BaseModel):
    value: int

class StrictModel(BaseModel):
    model_config = ConfigDict(strict=True)
    value: int

print(LooseModel(value="42"))

try:
    StrictModel(value="42")
except ValidationError as exc:
    print(exc)

## 字段验证器 field_validator

字段验证器用于单字段清洗和校验。

验证模式：

- `before`：在 Pydantic 类型处理前执行
- `after`：类型处理完成后执行，通常更安全
- `plain`：完全接管字段验证
- `wrap`：包裹 Pydantic 的内部验证流程

In [ ]:
from pydantic import field_validator

class Registration(BaseModel):
    username: str
    email: str
    interests: list[str]

    @field_validator("username")
    @classmethod
    def normalize_username(cls, value: str) -> str:
        value = value.strip().lower()
        if len(value) < 3:
            raise ValueError("用户名长度不能少于 3")
        return value

    @field_validator("interests", mode="before")
    @classmethod
    def split_interests(cls, value: object) -> object:
        if isinstance(value, str):
            return [item.strip() for item in value.split(",") if item.strip()]
        return value

registration = Registration(
    username="  Alice  ",
    email="alice@example.com",
    interests="Python, Pydantic, API",
)

print(registration)

## 模型验证器 model_validator

模型验证器适合跨字段规则，例如：

- 开始时间必须早于结束时间
- 密码与确认密码必须一致
- 折扣价不能高于原价

In [ ]:
from pydantic import model_validator

class Discount(BaseModel):
    original_price: Annotated[float, Field(gt=0)]
    sale_price: Annotated[float, Field(gt=0)]

    @model_validator(mode="after")
    def check_price(self) -> "Discount":
        if self.sale_price > self.original_price:
            raise ValueError("促销价不能高于原价")
        return self

print(Discount(original_price=100, sale_price=80))

try:
    Discount(original_price=100, sale_price=120)
except ValidationError as exc:
    print(exc)

## computed_field 计算字段

计算字段不需要用户输入，而是根据其他字段动态计算，并可参与序列化。

In [ ]:
from pydantic import computed_field

class Rectangle(BaseModel):
    width: Annotated[float, Field(gt=0)]
    height: Annotated[float, Field(gt=0)]

    @computed_field
    @property
    def area(self) -> float:
        return self.width * self.height

rectangle = Rectangle(width=4, height=5)
print(rectangle.area)
print(rectangle.model_dump())

## 字段别名

字段别名常用于：

- Python 使用 snake_case
- 外部 JSON 使用 camelCase
- 对接历史接口字段

In [ ]:
class ApiUser(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    user_id: int = Field(alias="userId")
    display_name: str = Field(alias="displayName")

user1 = ApiUser(userId=1, displayName="Alice")
user2 = ApiUser(user_id=2, display_name="Bob")

print(user1.model_dump())
print(user1.model_dump(by_alias=True))
print(user2)

## Enum 与 Literal 的选择

- `Literal`：选项少、只关心字面量时简单直接
- `Enum`：需要复用、附加方法或统一管理时更合适

In [ ]:
from enum import Enum

class OrderStatus(str, Enum):
    CREATED = "created"
    PAID = "paid"
    SHIPPED = "shipped"
    CLOSED = "closed"

class Order(BaseModel):
    id: int
    status: OrderStatus = OrderStatus.CREATED

order = Order(id=1, status="paid")
print(order)
print(order.model_dump())
print(order.model_dump_json())

## 日期、时间与 UUID

Pydantic 可以根据类型标注解析常见标准格式。

In [ ]:
from datetime import date, datetime
from uuid import UUID

class Event(BaseModel):
    event_id: UUID
    title: str
    event_date: date
    created_at: datetime

event = Event(
    event_id="12345678-1234-5678-1234-567812345678",
    title="Pydantic Workshop",
    event_date="2026-08-01",
    created_at="2026-07-27T10:30:00+09:00",
)

print(event)
print(type(event.event_id))
print(type(event.event_date))
print(type(event.created_at))

## EmailStr 等扩展类型

`EmailStr` 需要安装 `email-validator`。它适合做格式校验，但不代表邮箱一定真实存在。

In [ ]:
from pydantic import EmailStr

class Contact(BaseModel):
    name: str
    email: EmailStr

print(Contact(name="Alice", email="alice@example.com"))

try:
    Contact(name="Alice", email="not-an-email")
except ValidationError as exc:
    print(exc)

## TypeAdapter：验证非模型类型

当目标只是 `list[int]`、联合类型、`TypedDict` 或其他类型表达式时，不一定要专门创建 `BaseModel`。

In [ ]:
from pydantic import TypeAdapter

int_list_adapter = TypeAdapter(list[int])

values = int_list_adapter.validate_python(["1", 2, 3])
print(values)
print(int_list_adapter.dump_json(values))

UserIdOrName = int | str
id_adapter = TypeAdapter(UserIdOrName)
print(id_adapter.validate_python(100))
print(id_adapter.validate_python("alice"))

## RootModel：根对象不是字典时

某些 JSON 顶层结构直接是列表、字符串或映射。此时可使用 `RootModel`。

In [ ]:
from pydantic import RootModel

class UserIdList(RootModel[list[int]]):
    pass

ids = UserIdList.model_validate(["1", 2, 3])
print(ids)
print(ids.root)
print(ids.model_dump())

## Pydantic 泛型模型

泛型模型可以统一描述分页响应、API 响应和结果容器。

In [ ]:
class ApiResponse(BaseModel, Generic[T]):
    success: bool
    data: T | None = None
    message: str | None = None

class UserInfo(BaseModel):
    id: int
    name: str

response = ApiResponse[UserInfo](
    success=True,
    data={"id": 1, "name": "Alice"},
)

print(response)
print(type(response.data))

## 递归模型与前向引用

树、菜单、评论回复和组织结构通常需要模型引用自身。

In [ ]:
class TreeNode(BaseModel):
    name: str
    children: list["TreeNode"] = Field(default_factory=list)

tree = TreeNode(
    name="root",
    children=[
        {"name": "docs"},
        {
            "name": "src",
            "children": [
                {"name": "main.py"},
                {"name": "models.py"},
            ],
        },
    ],
)

print(tree.model_dump())

## 判别联合 Discriminated Union

当多个模型共享一个类型字段时，可以让 Pydantic 根据该字段选择具体模型，适合事件流、支付方式和多形态配置。

In [ ]:
class EmailNotification(BaseModel):
    type: Literal["email"]
    email: EmailStr
    subject: str

class SmsNotification(BaseModel):
    type: Literal["sms"]
    phone: str
    content: str

Notification = Annotated[
    EmailNotification | SmsNotification,
    Field(discriminator="type"),
]

class NotificationTask(BaseModel):
    notification: Notification

email_task = NotificationTask(
    notification={
        "type": "email",
        "email": "alice@example.com",
        "subject": "Welcome",
    }
)

sms_task = NotificationTask(
    notification={
        "type": "sms",
        "phone": "13800000000",
        "content": "验证码 123456",
    }
)

print(email_task)
print(type(email_task.notification))
print(type(sms_task.notification))

## validate_call：验证函数参数

`validate_call` 可以将 Pydantic 验证用于普通函数边界。它适合脚本、服务层入口和工具函数，但不要滥用于高频内部小函数。

In [ ]:
from pydantic import validate_call

@validate_call
def create_order(
    user_id: int,
    product_ids: list[int],
    discount: Annotated[float, Field(ge=0, le=1)] = 0,
) -> dict[str, object]:
    return {
        "user_id": user_id,
        "product_ids": product_ids,
        "discount": discount,
    }

print(create_order("1001", ["1", "2"], discount="0.2"))

try:
    create_order(1001, [1, 2], discount=1.5)
except ValidationError as exc:
    print(exc)

## Pydantic dataclass

如果项目偏好 dataclass 风格，但仍希望运行时验证，可以使用 `pydantic.dataclasses.dataclass`。

In [ ]:
from pydantic.dataclasses import dataclass

@dataclass
class Point:
    x: int
    y: int

point = Point(x="10", y="20")
print(point)
print(type(point.x))

## JSON Schema

模型可以生成 JSON Schema，常用于：

- API 文档
- 表单生成
- 配置编辑器
- LLM 结构化输出
- 跨语言接口契约

In [ ]:
class CreateUserRequest(BaseModel):
    username: Annotated[
        str,
        Field(min_length=3, max_length=30, description="登录用户名"),
    ]
    age: Annotated[
        int,
        Field(ge=18, le=120, description="用户年龄"),
    ]
    email: EmailStr

schema = CreateUserRequest.model_json_schema()
print(schema)

# 综合实战：电商订单请求模型

本案例覆盖：

- 嵌套模型
- 枚举
- 字段约束
- 跨字段校验
- 计算字段
- 序列化

In [ ]:
from decimal import Decimal

class PaymentMethod(str, Enum):
    WECHAT = "wechat"
    ALIPAY = "alipay"
    CARD = "card"

class OrderItem(BaseModel):
    product_id: int
    product_name: Annotated[str, Field(min_length=1, max_length=100)]
    unit_price: Annotated[Decimal, Field(gt=0)]
    quantity: Annotated[int, Field(ge=1, le=999)]

    @computed_field
    @property
    def subtotal(self) -> Decimal:
        return self.unit_price * self.quantity

class CreateOrderRequest(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
    )

    user_id: int
    items: Annotated[list[OrderItem], Field(min_length=1)]
    payment_method: PaymentMethod
    coupon_amount: Annotated[Decimal, Field(ge=0)] = Decimal("0")
    remark: Annotated[str | None, Field(max_length=200)] = None

    @computed_field
    @property
    def original_amount(self) -> Decimal:
        return sum((item.subtotal for item in self.items), Decimal("0"))

    @computed_field
    @property
    def payable_amount(self) -> Decimal:
        return self.original_amount - self.coupon_amount

    @model_validator(mode="after")
    def validate_coupon(self) -> "CreateOrderRequest":
        if self.coupon_amount > self.original_amount:
            raise ValueError("优惠金额不能超过订单原始金额")
        return self

request = CreateOrderRequest.model_validate(
    {
        "user_id": "10001",
        "items": [
            {
                "product_id": 101,
                "product_name": "机械键盘",
                "unit_price": "499.00",
                "quantity": 1,
            },
            {
                "product_id": 102,
                "product_name": "鼠标垫",
                "unit_price": "39.90",
                "quantity": 2,
            },
        ],
        "payment_method": "alipay",
        "coupon_amount": "50.00",
    }
)

print(request)
print(request.model_dump())
print(request.model_dump_json(indent=2))

# 综合实战：应用配置模型

配置模型适合读取环境变量、YAML 或 JSON 后先统一验证，再传给业务模块。

In [ ]:
class DatabaseConfig(BaseModel):
    host: str = "localhost"
    port: Annotated[int, Field(ge=1, le=65535)] = 5432
    username: str
    password: str
    database: str
    pool_size: Annotated[int, Field(ge=1, le=100)] = 10

class ModelConfig(BaseModel):
    provider: Literal["openai", "anthropic", "local"]
    model_name: str
    temperature: Annotated[float, Field(ge=0, le=2)] = 0.7
    max_tokens: Annotated[int, Field(gt=0)] = 2048

class AppConfig(BaseModel):
    model_config = ConfigDict(extra="forbid")

    environment: Environment
    debug: bool = False
    database: DatabaseConfig
    llm: ModelConfig

config = AppConfig.model_validate(
    {
        "environment": "dev",
        "debug": True,
        "database": {
            "username": "app",
            "password": "secret",
            "database": "academy",
        },
        "llm": {
            "provider": "local",
            "model_name": "qwen",
            "temperature": 0.3,
        },
    }
)

print(config.model_dump(exclude={"database": {"password"}}))

# typing 与 Pydantic 的边界

| 工具 | 主要阶段 | 是否运行时验证 | 典型用途 |
|---|---|---:|---|
| 普通类型标注 | 开发期 | 否 | IDE、代码阅读、静态检查 |
| mypy / pyright | 开发期或 CI | 不直接改变运行 | 提前发现类型不一致 |
| TypedDict | 开发期 | 否 | 描述字典结构 |
| dataclass | 运行时对象 | 默认否 | 轻量数据对象 |
| Pydantic BaseModel | 运行时边界 | 是 | API、配置、外部数据 |
| TypeAdapter | 运行时边界 | 是 | 验证任意类型表达式 |
| Protocol | 开发期 | 否 | 描述行为接口 |

推荐原则：

- 内部可信数据：优先普通类型标注和 dataclass
- 外部不可信数据：使用 Pydantic
- 只验证一个类型表达式：使用 TypeAdapter
- 只描述字典给静态检查器：使用 TypedDict
- 跨模块行为契约：使用 Protocol

# 常见错误与修正

## 错误：把类型提示当作运行时验证

```python
def set_age(age: int) -> None:
    print(age)

set_age("18")  # Python 可能仍会执行
```

修正方式：

- 在 CI 中运行 mypy 或 pyright
- 在外部数据边界使用 Pydantic
- 不要期待类型标注自动抛出运行时异常

## 错误：Optional 字段没有默认值

```python
class User(BaseModel):
    nickname: str | None
```

这表示值允许是 `None`，但字段仍然必须提供。若字段可省略：

```python
class User(BaseModel):
    nickname: str | None = None
```

## 错误：验证器忘记返回值

字段验证器和模型验证器通常必须返回处理后的值或模型。

## 错误：在模型中塞入过多业务逻辑

Pydantic 适合边界验证与轻量派生。数据库查询、网络调用和复杂流程应放在服务层。

## 错误：默认允许多余字段

关键配置、支付、权限等模型可设置 `extra="forbid"`，避免拼写错误被静默忽略。

## 错误：盲目开启严格模式

严格模式更安全，但也可能让正常的 HTTP、表单和环境变量字符串输入无法转换。应根据输入来源决定。

# 工程最佳实践

## 模型分层

可以按职责区分：

- `CreateUserRequest`：创建请求
- `UpdateUserRequest`：更新请求
- `UserResponse`：对外响应
- `UserEntity`：数据库或领域实体
- `UserConfig`：配置模型

不要让一个模型同时承担数据库、API、业务和展示的全部职责。

## 输入与输出分离

输入中可能包含密码，输出模型不应包含密码字段。

## 约束靠近类型

推荐：

```python
UserId = Annotated[int, Field(gt=0)]
UserName = Annotated[str, Field(min_length=3, max_length=30)]
```

这样可在多个模型中复用。

## 错误转换

API 层可将 `ValidationError.errors()` 转换为统一错误响应，但内部日志应保留原始上下文。

## 静态检查与运行时验证同时使用

Pydantic 不能替代 mypy，mypy 也不能替代 Pydantic：

- mypy 检查代码之间的类型一致性
- Pydantic 检查运行时进入系统的数据

## mypy 示例

保存为 `example.py`：

```python
def total(values: list[int]) -> int:
    return sum(values)

result: str = total([1, 2, 3])
```

运行：

```bash
mypy example.py
```

静态检查器会指出 `int` 不能赋值给 `str`。

# 总结

学习路线可以概括为：

1. 先掌握函数、变量和容器的基础类型标注
2. 掌握 `Union`、`Literal`、`TypedDict`、`Protocol` 和泛型
3. 使用 `Annotated` 表达可复用约束
4. 使用 Pydantic `BaseModel` 验证外部数据
5. 掌握 `Field`、验证器、配置和序列化
6. 使用 `TypeAdapter` 处理非模型类型
7. 使用泛型、判别联合和递归模型处理复杂结构
8. 将静态类型检查与运行时验证结合到 CI 和服务边界

参考资料：

- Python typing 官方文档：https://docs.python.org/3/library/typing.html
- Pydantic 官方文档：https://docs.pydantic.dev/latest/